# Notebook 01 mini - AmazonLike prototype intersection

Questo notebook crea una versione piccola,varia e bilanciata del dataset Amazon-like per alimentare il sito prototipo.

Interseca tre sorgenti:

- metadata Amazon Reviews 2023;
- QA storico McAuley;
- rating/reviews Amazon Reviews 2023.

Output principali:

```text
prototype/data/processed/prototype_amazon_like_intersection/
  items.parquet
  qa.parquet
  ratings.parquet
  category_summary.csv
  summary.json

prototype/data/products_demo.json
```

Non modifica i notebook originali e non scrive in `data/raw/` o `data/processed/`, salvo lettura dei QA storici gia presenti/scaricabili.

In [1]:
# Se mancano pacchetti, esegui questa cella una sola volta.
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path #per gestire meglio percorsi di file e cartelle
import ast #servir? pi? avanti per leggere file del QA storico
import gzip #leggere e scrivere file compressi gz
import json #lavorare con json
import math
from typing import Iterable,Optional #tipi

import pandas as pd #pandas per creare dataframe
import pyarrow as pa #per definire schemi tabellari
import pyarrow.parquet as pq #per leggere metadata dei file .parquet
import requests #per scaricare da URL
from tqdm.auto import tqdm #barra di progresso
import datasets #pacchetto datasets
from datasets import load_dataset #per caricare dataset

datasets.logging.set_verbosity_error() #riduce i messaggi stampati da datasets


c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#configurazione
DATASET_NAME="prototype_amazon_like_intersection"
DATASET_LABEL="AmazonLikePrototypeMini"

#label ? il nome della categoria che usiamo noi
#hfcategory ? il nome usato su amazon reviews
#qaslug ? il nome del file qa di quella categoria
#group macro categoria creata da me
CATEGORIES=[
    {"label":"Appliances","hf_category":"Appliances","qa_slug":"Appliances","group":"home"},
    {"label":"Arts Crafts and Sewing","hf_category":"Arts_Crafts_and_Sewing","qa_slug":"Arts_Crafts_and_Sewing","group":"crafts"},
    {"label":"Automotive","hf_category":"Automotive","qa_slug":"Automotive","group":"automotive"},
    {"label":"Baby Products","hf_category":"Baby_Products","qa_slug":"Baby","group":"baby"},
    {"label":"Beauty","hf_category":"All_Beauty","qa_slug":"Beauty","group":"beauty"},
    {"label":"Cell Phones and Accessories","hf_category":"Cell_Phones_and_Accessories","qa_slug":"Cell_Phones_and_Accessories","group":"tech"},
    {"label":"Clothing Shoes and Jewelry","hf_category":"Clothing_Shoes_and_Jewelry","qa_slug":"Clothing_Shoes_and_Jewelry","group":"fashion"},
    {"label":"Electronics","hf_category":"Electronics","qa_slug":"Electronics","group":"tech"},
    {"label":"Grocery and Gourmet Food","hf_category":"Grocery_and_Gourmet_Food","qa_slug":"Grocery_and_Gourmet_Food","group":"grocery"},
    {"label":"Health and Personal Care","hf_category":"Health_and_Personal_Care","qa_slug":"Health_and_Personal_Care","group":"health"},
    {"label":"Home and Kitchen","hf_category":"Home_and_Kitchen","qa_slug":"Home_and_Kitchen","group":"home"},
    {"label":"Industrial and Scientific","hf_category":"Industrial_and_Scientific","qa_slug":"Industrial_and_Scientific","group":"industrial"},
    {"label":"Musical Instruments","hf_category":"Musical_Instruments","qa_slug":"Musical_Instruments","group":"music"},
    {"label":"Office Products","hf_category":"Office_Products","qa_slug":"Office_Products","group":"office"},
    {"label":"Patio Lawn and Garden","hf_category":"Patio_Lawn_and_Garden","qa_slug":"Patio_Lawn_and_Garden","group":"garden"},
    {"label":"Pet Supplies","hf_category":"Pet_Supplies","qa_slug":"Pet_Supplies","group":"pets"},
    {"label":"Software","hf_category":"Software","qa_slug":"Software","group":"tech"},
    {"label":"Sports and Outdoors","hf_category":"Sports_and_Outdoors","qa_slug":"Sports_and_Outdoors","group":"sports"},
    {"label":"Tools and Home Improvement","hf_category":"Tools_and_Home_Improvement","qa_slug":"Tools_and_Home_Improvement","group":"tools"},
    {"label":"Toys and Games","hf_category":"Toys_and_Games","qa_slug":"Toys_and_Games","group":"toys"},
    {"label":"Video Games","hf_category":"Video_Games","qa_slug":"Video_Games","group":"gaming"},
]

#nota mapping Beauty
#se Beauty produce 0 prodotti o problemi,provare hf_category="Beauty_and_Personal_Care" come alternativa.

QA_BASE_URL="https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa"
HF_DATASET_BASE_URL="https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main"
SAVE_TEXT_OF_REVIEW=True
FORCE_REBUILD_RATINGS=False


In [4]:
#Limiti medi per creare un dataset prototipo piu realistico ma ancora gestibile
#Gli item restano scelti manualmente per controllare il bilanciamento tra categorie
#Metadata, reviews e rating vengono invece calcolati in proporzione agli item

MAX_ITEMS_BY_CATEGORY={
    "Electronics":1000,
    "Cell Phones and Accessories":500,
    "Home and Kitchen":1000,
    "Sports and Outdoors":1000,
    "Appliances":500,
    "Arts Crafts and Sewing":100,
    "Automotive":1000,
    "Baby Products":500,
    "Beauty":500,
    "Clothing Shoes and Jewelry":1000,
    "Grocery and Gourmet Food":500,
    "Health and Personal Care":1000,
    "Industrial and Scientific":500,
    "Musical Instruments":200,
    "Office Products":1000,
    "Patio Lawn and Garden":200,
    "Pet Supplies":100,
    "Software":100,
    "Tools and Home Improvement":500,
    "Toys and Games":500,
    "Video Games":500,
}

#Parametri generali per generare limiti coerenti
#I valori minimi evitano categorie troppo piccole o quasi vuote
#I valori massimi evitano scansioni e file finali troppo pesanti

METADATA_SCAN_MULTIPLIER=250
REVIEWS_SCAN_MULTIPLIER=350
RATINGS_PER_ITEM_TARGET=25

MIN_METADATA_SCAN=80000
MAX_METADATA_SCAN=300000

MIN_REVIEWS_SCAN=100000
MAX_REVIEWS_SCAN=400000

MIN_RATINGS_PER_CATEGORY=3000
MAX_RATINGS_PER_CATEGORY=30000

#Limiti di scansione metadata Amazon Reviews 2023
#Scansioniamo piu righe degli item finali per aumentare la probabilita di trovare prodotti utili
#Il limite cresce con il numero di item richiesti per categoria

MAX_METADATA_SCAN_BY_CATEGORY={
    category:int(min(
        max(item_limit*METADATA_SCAN_MULTIPLIER,MIN_METADATA_SCAN),
        MAX_METADATA_SCAN
    ))
    for category,item_limit in MAX_ITEMS_BY_CATEGORY.items()
}

#Limiti di scansione review Amazon Reviews 2023
#Le review vengono scansionate piu largamente per trovare abbastanza rating sui prodotti selezionati
#Il limite resta comunque controllato per non rallentare troppo il notebook

MAX_REVIEWS_SCAN_BY_CATEGORY={
    category:int(min(
        max(item_limit*REVIEWS_SCAN_MULTIPLIER,MIN_REVIEWS_SCAN),
        MAX_REVIEWS_SCAN
    ))
    for category,item_limit in MAX_ITEMS_BY_CATEGORY.items()
}

#Limiti di rating salvati per categoria
#I rating salvati sono proporzionali agli item invece di essere quasi uguali per tutte le categorie
#Questo evita categorie grandi troppo sparse e categorie piccole troppo sovrarappresentate

MAX_RATINGS_SAVED_BY_CATEGORY={
    category:int(min(
        max(item_limit*RATINGS_PER_ITEM_TARGET,MIN_RATINGS_PER_CATEGORY),
        MAX_RATINGS_PER_CATEGORY
    ))
    for category,item_limit in MAX_ITEMS_BY_CATEGORY.items()
}

#Controlliamo che tutti i dizionari abbiano le stesse categorie
#Se un dizionario non combacia, il notebook potrebbe usare limiti mancanti
#Questi assert bloccano subito errori di configurazione

assert set(MAX_METADATA_SCAN_BY_CATEGORY)==set(MAX_ITEMS_BY_CATEGORY)
assert set(MAX_REVIEWS_SCAN_BY_CATEGORY)==set(MAX_ITEMS_BY_CATEGORY)
assert set(MAX_RATINGS_SAVED_BY_CATEGORY)==set(MAX_ITEMS_BY_CATEGORY)

print("Totale item target:",sum(MAX_ITEMS_BY_CATEGORY.values()))
print("Totale metadata scan massimo:",sum(MAX_METADATA_SCAN_BY_CATEGORY.values()))
print("Totale review scan massimo:",sum(MAX_REVIEWS_SCAN_BY_CATEGORY.values()))
print("Totale rating salvati massimo:",sum(MAX_RATINGS_SAVED_BY_CATEGORY.values()))

Totale item target: 12200
Totale metadata scan massimo: 3275000
Totale review scan massimo: 4525000
Totale rating salvati massimo: 306500


In [5]:
#percorsi assoluti della root del repository
PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
PROTOTYPE_DIR=PROJECT_ROOT/"prototype"
RAW_DIR=PROJECT_ROOT/"data"/"raw"
QA_RAW_DIR=RAW_DIR/"qa"
QA_WORK_DIR=PROTOTYPE_DIR/"data"/"raw"/"qa"
PROCESSED_DIR=PROTOTYPE_DIR/"data"/"processed"/DATASET_NAME
PRODUCTS_DEMO_PATH=PROTOTYPE_DIR/"data"/"products_demo.json"

#percorsi output notebook
ITEMS_OUT=PROCESSED_DIR/"items.parquet" #percorso finale dei prodotti
QA_OUT=PROCESSED_DIR/"qa.parquet" #percorso file domande/risposte
RATINGS_OUT=PROCESSED_DIR/"ratings.parquet" #percorso rating
RATINGS_BUILD_INFO_PATH=PROCESSED_DIR/"ratings_build_info.json" #firma configurazione usata per ratings.parquet
CATEGORY_SUMMARY_PATH=PROCESSED_DIR/"category_summary.csv" #riepilogo per categoria
SUMMARY_PATH=PROCESSED_DIR/"summary.json" #riepilogo generale

#creiamo solo cartelle del prototipo
PROCESSED_DIR.mkdir(parents=True,exist_ok=True)
PRODUCTS_DEMO_PATH.parent.mkdir(parents=True,exist_ok=True)
QA_WORK_DIR.mkdir(parents=True,exist_ok=True)

print("Project root:",PROJECT_ROOT)
print("Output processed:",PROCESSED_DIR)
print("Output demo catalog:",PRODUCTS_DEMO_PATH)
print("Categorie selezionate:")
for category in CATEGORIES:
    print("-",category["label"],"| hf=",category["hf_category"],"| qa=",category["qa_slug"])


Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Output processed: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\processed\prototype_amazon_like_intersection
Output demo catalog: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\products_demo.json
Categorie selezionate:
- Appliances | hf= Appliances | qa= Appliances
- Arts Crafts and Sewing | hf= Arts_Crafts_and_Sewing | qa= Arts_Crafts_and_Sewing
- Automotive | hf= Automotive | qa= Automotive
- Baby Products | hf= Baby_Products | qa= Baby
- Beauty | hf= All_Beauty | qa= Beauty
- Cell Phones and Accessories | hf= Cell_Phones_and_Accessories | qa= Cell_Phones_and_Accessories
- Clothing Shoes and Jewelry | hf= Clothing_Shoes_and_Jewelry | qa= Clothing_Shoes_and_Jewelry
- Electronics | hf= Electronics | qa= Electronics
- Grocery and Gourmet Food | hf= Grocery_and_Gourmet_Food | qa= Grocery_and_Gourmet_Food
- Health and Personal Care | hf= Health_and_Personal_Care | qa= Health_and_Personal_Care
- Home and Kitchen | hf= Home_an

In [6]:
#Scarichiamo file esterni in modo streaming
#Creiamo cartelle prima del download
#Il rischio principale e interrompere download grandi
#scarica da url in output_path
def download_file(url:str,output_path:Path,force:bool=False)->Path:
    if output_path.exists() and not force: #non riscarica
        print(f"Gia presente: {output_path}")
        return output_path

    output_path.parent.mkdir(parents=True,exist_ok=True) #se non esiste cartella di destinazione la crea
    with requests.get(url,stream=True,timeout=60) as response:
        response.raise_for_status()
        total=int(response.headers.get("content-length",0)) #content-length puo mancare,0 evita errore
        with open(output_path,"wb") as file,tqdm(total=total,unit="B",unit_scale=True,desc=output_path.name) as pbar: #apre file in scrittura binaria, perch? scarichi .gz e apre barra tqdm
            for chunk in response.iter_content(chunk_size=1024*1024): #scarica blocchi da 1MB senza caricare tutto in RAM
                if chunk:
                    file.write(chunk)
                    pbar.update(len(chunk))
    return output_path

#legge qa storico, restituisce dizionari python
def iter_historical_qa(path:Path)->Iterable[dict]:
    with gzip.open(path,"rt",encoding="utf-8",errors="replace") as file: #apre il file .gz
        for line in file:
            line=line.strip() #rimuove spazi e newline
            if line:
                yield ast.literal_eval(line) #converte riga da testo a dizionario python

#legge file .jsonl.gz
def iter_jsonl_gz(path:Path)->Iterable[dict]:
    with gzip.open(path,"rt",encoding="utf-8",errors="replace") as file: #apre il file .gz
        for line in file:
            line=line.strip() #rimuove spazi e newline
            if line:
                yield json.loads(line)

#Leggiamo JSONL remoto riga per riga
#Evitiamo download completo quando basta filtrare in streaming
#Il timeout lungo serve per dataset grandi
#legge un file jsonl da Hugging Face senza scaricarlo tutto prima
def iter_hf_jsonl(relative_path:str)->Iterable[dict]:
    url=f"{HF_DATASET_BASE_URL}/{relative_path}"
    with requests.get(url,stream=True,timeout=(30,300)) as response: #timeout separato connessione/lettura
        response.raise_for_status()
        for line in response.iter_lines(chunk_size=1024*1024):
            if line:
                yield json.loads(line)


In [7]:
#Estraiamo immagine principale da strutture diverse
#Amazon puo salvare immagini come dict o lista di dict
#Teniamo il primo URL disponibile per il catalogo demo
#prova a estrarre immagine da campo images
def image_url(images)->Optional[str]:
    if isinstance(images,dict):
        for key in ("hi_res","large","thumb","large_image_url","medium_image_url","small_image_url"): #ordine di preferenza qualita immagine
            values=images.get(key) or []
            if isinstance(values,str):
                values=[values]
            for value in values:
                if value:
                    return value
    if isinstance(images,list):
        for image in images:
            if isinstance(image,dict):
                for key in ("hi_res","large","thumb","large_image_url","medium_image_url","small_image_url"):
                    value=image.get(key)
                    if value:
                        return value
    return None

#Convertiamo valori annidati in JSON testuale
#Serve per salvare liste/dict dentro parquet senza perdere struttura
#Manteniamo ensure_ascii=False per testo leggibile
#converte valore in stringa json
def tojson_string(value)->str:
    if value is None:
        return "null"
    return json.dumps(value,ensure_ascii=False)

#converte valore in numero decimale
def tofloat_or_none(value):
    try:
        if value in (None,"","None"):
            return None
        return float(value)
    except Exception:
        return None

#converte in intero
def toint_or_none(value):
    try:
        if value in (None,"","None"):
            return None
        return int(value)
    except Exception:
        return None

#recuperiamo categoria prodotto dai metadata
def get_category(row:dict,default_category:str)->Optional[str]:
    main_category=row.get("main_category")
    if main_category not in (None,"","None"):
        return main_category

    categories=row.get("categories")
    if isinstance(categories,list) and len(categories)>0:
        return categories[0]

    return default_category

#Scriviamo righe in parquet in modo incrementale
#Il writer resta aperto tra batch per evitare file multipli
#La schema garantisce colonne e tipi coerenti
#scrive righe in un file parquet, rows=listadiz, path=dove salvare, schema=schema PyArrow, writer=oggetto che gestisce la scrittura Parquet
def writerows_in_parquet(rows:list[dict],path:Path,schema:pa.Schema,writer:Optional[pq.ParquetWriter]):
    if not rows:
        return writer

    df=pd.DataFrame(rows) #converte lista di dizionari in un dataframe pandas
    table=pa.Table.from_pandas(df,schema=schema,preserve_index=False) #converte dataframe in tabella pyarrow
    if writer is None:
        path.parent.mkdir(parents=True,exist_ok=True) #crea cartella di output
        writer=pq.ParquetWriter(path,schema=schema,compression="zstd") #crea writer solo al primo batch
    writer.write_table(table)
    return writer

#funzione che conta righe di un file parquet
def parquet_count_rows(path:Path)->int:
    return pq.ParquetFile(path).metadata.num_rows


In [8]:
#Convertiamo QA storico da JSONL gzip a JSON e aggiungiamo parent_asin per
#agganciarlo al dataset amazon 2023
#per fare questo l'asin storico lo usiamo come parent_asin
#dizionari per info per ogni categoria
qa_info_by_category={}
category_summary_rows=[]

for category in CATEGORIES: #cicliamo ogni categoria
    qa_slug=category["qa_slug"]
    qa_url=f"{QA_BASE_URL}/qa_{qa_slug}.json.gz"
    qa_source_path=QA_RAW_DIR/f"qa_{qa_slug}.json.gz"
    qa_cache_path=QA_WORK_DIR/f"qa_{qa_slug}.json.gz"
    qa_strict_path=QA_WORK_DIR/f"qa_{qa_slug}.strict.jsonl.gz"

    #file originale scaricato e file convertito
    qa_raw_path=qa_source_path if qa_source_path.exists() else download_file(qa_url,qa_cache_path)

    qa_asins=set()
    qa_rows=0

    with gzip.open(qa_strict_path,"wt",encoding="utf-8") as output:
        for qa_id,row in enumerate(tqdm(iter_historical_qa(qa_raw_path),desc=f"Conversione QA {category['label']}")):
            asin=row.get("asin") #prende valore dal campo asin
            row["qa_id"]=qa_id #aggiunge id numerico alla riga qa nella categoria
            row["parent_asin"]=asin #crea il campo parent_asin e ci copia l'asin
            row["source_category"]=category["label"] #categoria
            row["hf_category"]=category["hf_category"] #categoria huggingface
            row["qa_slug"]=category["qa_slug"] #file qa da cui arriva
            row["category_group"]=category["group"] #macrogruppo

            if asin:
                qa_asins.add(asin)

            output.write(json.dumps(row,ensure_ascii=False)+"\n") #scrive la riga nel nuovo file jsonl
            qa_rows+=1

    qa_info_by_category[category["label"]]={ #info qa per categoria
        "qa_raw_path":qa_raw_path,
        "qa_strict_path":qa_strict_path,
        "qa_asins":qa_asins,
        "qa_rows":qa_rows,
        "qa_products":len(qa_asins),
    }
    category_summary_rows.append({ #riepilogo senza asin, pi? leggero
        "source_category":category["label"],
        "hf_category":category["hf_category"],
        "qa_slug":category["qa_slug"],
        "category_group":category["group"],
        "qa_rows":qa_rows,
        "qa_products":len(qa_asins),
    })

    print(category["label"],"| QA rows:",qa_rows,"| prodotti QA:",len(qa_asins))


Conversione QA Appliances: 9011it [00:00, 22414.61it/s]


Appliances | QA rows: 9011 | prodotti QA: 1153


Conversione QA Arts Crafts and Sewing: 21262it [00:00, 23584.24it/s]


Arts Crafts and Sewing | QA rows: 21262 | prodotti QA: 2944


Conversione QA Automotive: 89923it [00:03, 23410.14it/s]


Automotive | QA rows: 89923 | prodotti QA: 12536


... (32 outputs left out)

Conversione QA Toys and Games: 51486it [00:01, 25792.71it/s]


Toys and Games | QA rows: 51486 | prodotti QA: 7393


Conversione QA Video Games: 13307it [00:00, 28585.79it/s]

Video Games | QA rows: 13307 | prodotti QA: 1188


In [9]:
#Prepariamo schema e funzioni per i metadati prodotto
#Teniamo solo prodotti presenti nel QA e nei metadata Amazon 2023
#Deduplichiamo parent_asin fra categorie per evitare doppioni
#prepariamo funzioni comuni e schema dei prodotti
#ogni riga salvata mantiene anche source_category per analisi multi-categoria
#schema del file items.parquet
ITEM_SCHEMA=pa.schema([
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("title",pa.string()),
    ("category",pa.string()),
    ("avg_rating",pa.float64()),
    ("rating_number",pa.int64()),
    ("price",pa.float64()),
    ("store",pa.string()),
    ("features_json",pa.string()),
    ("description_json",pa.string()),
    ("categories_json",pa.string()),
    ("details_json",pa.string()),
    ("image_url",pa.string()),
])

#se items.parquet esiste gi? lo elimina
if ITEMS_OUT.exists():
    ITEMS_OUT.unlink()

writer=None
buffer=[]
global_items_seen=set()
items_id_by_category={}

#carichiamo metadati amazon reviews 2023 da Hugging Face per ogni categoria
#teniamo solo prodotti presenti anche nel QA e deduplichiamo parent_asin fra categorie
#creaiamo intes.parquet cio? tab prodotti finali
#leggiamo direttamente i file JSONL riga per riga, cos? possiamo fermarci prima
for category in CATEGORIES: #cicla ogni categoria
    label=category["label"]
    hf_category=category["hf_category"]
    qa_asins=qa_info_by_category[label]["qa_asins"]
    meta_path=f"raw/meta_categories/meta_{hf_category}.jsonl"

    matched_items=0
    duplicates_skipped=0
    category_items=set()
    metadata_rows_scanned=0
    max_items=MAX_ITEMS_BY_CATEGORY.get(label)
    max_metadata_scan=MAX_METADATA_SCAN_BY_CATEGORY.get(label)

    for row in tqdm(iter_hf_jsonl(meta_path),desc=f"Filtro metadati {label}"):
        if max_metadata_scan is not None and metadata_rows_scanned>=max_metadata_scan:
            break
        metadata_rows_scanned+=1

        parent_asin=row.get("parent_asin")
        if parent_asin not in qa_asins: #salta items non in qa storico
            continue
        if parent_asin in global_items_seen: #salta prodotto se compare in pi? categorie, senza duplicati
            duplicates_skipped+=1
            continue

        buffer.append({ #ggiunge prodotto al buffer
            "parent_asin":parent_asin,
            "source_category":label,
            "hf_category":hf_category,
            "qa_slug":category["qa_slug"],
            "category_group":category["group"],
            "title":row.get("title"),
            "category":get_category(row,label),
            "avg_rating":tofloat_or_none(row.get("average_rating")),
            "rating_number":toint_or_none(row.get("rating_number")),
            "price":tofloat_or_none(row.get("price")),
            "store":row.get("store"),
            "features_json":tojson_string(row.get("features")),
            "description_json":tojson_string(row.get("description")),
            "categories_json":tojson_string(row.get("categories")),
            "details_json":tojson_string(row.get("details")),
            "image_url":image_url(row.get("images")),
        })

        matched_items+=1
        category_items.add(parent_asin)
        global_items_seen.add(parent_asin)

        if max_items is not None and matched_items>=max_items:
            break
        if len(buffer)>=5000: #scrive batch per non tenere troppe righe in memoria
            writer=writerows_in_parquet(buffer,ITEMS_OUT,ITEM_SCHEMA,writer)
            buffer=[]

    items_id_by_category[label]=category_items #salva insieme dei prodotti di quella categoria
    for summary_row in category_summary_rows: #aggiorniamo riepilogo per categoria, con dati
        if summary_row["source_category"]==label:
            summary_row["metadata_rows"]=int(metadata_rows_scanned)
            summary_row["items_qa_metadata"]=int(matched_items)
            summary_row["duplicate_items_skipped"]=int(duplicates_skipped)
            break

    print(label,"| metadati scansionati:",metadata_rows_scanned,"| prodotti salvati:",matched_items,"| duplicati:",duplicates_skipped)

writer=writerows_in_parquet(buffer,ITEMS_OUT,ITEM_SCHEMA,writer)
if writer is not None:
    writer.close()

items_df=pd.read_parquet(ITEMS_OUT) if ITEMS_OUT.exists() else pd.DataFrame(columns=[field.name for field in ITEM_SCHEMA])
items_id=set(items_df["parent_asin"]) if len(items_df) else set()

print("Prodotti QA + metadata totali:",len(items_id))
display(items_df.head())


Filtro metadati Appliances: 67423it [00:06, 9736.73it/s] 


Appliances | metadati scansionati: 67424 | prodotti salvati: 500 | duplicati: 0


Filtro metadati Arts Crafts and Sewing: 58188it [00:06, 8990.46it/s] 


Arts Crafts and Sewing | metadati scansionati: 58189 | prodotti salvati: 100 | duplicati: 0


Filtro metadati Automotive: 218636it [00:16, 13102.08it/s]


Automotive | metadati scansionati: 218637 | prodotti salvati: 1000 | duplicati: 0


... (33 outputs left out)

Toys and Games | metadati scansionati: 123672 | prodotti salvati: 500 | duplicati: 0


Filtro metadati Video Games: 98666it [00:10, 9773.42it/s] 


Video Games | metadati scansionati: 98667 | prodotti salvati: 500 | duplicati: 0
Prodotti QA + metadata totali: 8999


,parent_asin,source_category,hf_category,qa_slug,category_group,title,category,avg_rating,rating_number,price,store,features_json,description_json,categories_json,details_json,image_url
0,B005H47EJ4,Appliances,Appliances,Appliances,home,"GE CGS985SETSS Cafe 30"" Stainless Steel Gas Se...",Appliances,3.1,30,NaN,GE,"[""Circulates heat for high-quality results"", ""...","[""Cafe 30\"" Stainless Steel Gas Sealed Burner ...","[""Appliances"", ""Ranges, Ovens & Cooktops"", ""Ra...","{""Brand Name"": ""GE"", ""Model Info"": ""CGS985SET""...",https://m.media-amazon.com/images/I/61vpWcONkJ...
1,B0057YO0IM,Appliances,Appliances,Appliances,home,"Bosch SHE3AR52UC Ascenta 24"" White Full Consol...",Appliances,3.0,47,NaN,BOSCH,"[""14 Place Settings"", ""4 Wash Cycles featuring...","[""24 '' Recessed Handle Dishwasher Ascenta Ser...","[""Appliances"", ""Dishwashers"", ""Built-In Dishwa...","{""Brand Name"": ""BOSCH"", ""Model Info"": ""SHE3AR5...",https://m.media-amazon.com/images/I/31y26u6wzB...
2,B001TIYKGC,Appliances,Appliances,Appliances,home,GARP 5300622034 Compatible Replacement for Dry...,Tools & Home Improvement,4.5,657,17.5,GARP,"[""Researched and Developed in New Jersey"", ""Li...","[""Product Description"", ""GARP products are res...","[""Appliances"", ""Parts & Accessories"", ""Dryer P...","{""Manufacturer"": ""GARP"", ""Part Number"": ""53006...",https://m.media-amazon.com/images/I/71faxojvw7...
3,B004OLUEZ2,Appliances,Appliances,Appliances,home,"Empava 30"" Built-in Electric Induction Cooktop...",Appliances,4.3,117,NaN,Whirlpool,"[""The USA & Canada ETL Certified Electric Stov...","[""The 30-in 4 elements black induction cooktop...","[""Appliances"", ""Ranges, Ovens & Cooktops"", ""Co...","{""Brand Name"": ""Whirlpool"", ""Model Info"": ""30\...",https://m.media-amazon.com/images/I/51SW30Ws4z...
4,B0002GTTRC,Appliances,Appliances,Appliances,home,GE GSWF Refrigerator Water Filter | Certified ...,Tools & Home Improvement,4.6,2030,40.0,General Electric,"[""Premium filtration: Certified to reduce part...","[""GSWF Features: -Water filter. -GE GSWF Refri...","[""Appliances"", ""Parts & Accessories"", ""Refrige...","{""Material"": ""water filter"", ""Product Dimensio...",https://m.media-amazon.com/images/I/71zwqUocXZ...


In [10]:
#Filtriamo QA sui soli prodotti presenti nei metadati 2023
#Cosi il modulo QA e il catalogo lavorano sugli stessi item
#Le righe fuori intersezione vengono scartate
# ora filtriamo QA e teniamo solo domande/risposte dei prodotti presenti anche nei metadati 2023
#schema del file qa.parquet
QA_SCHEMA=pa.schema([
    ("qa_id",pa.int64()),
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("question",pa.string()),
    ("answer",pa.string()),
    ("questionType",pa.string()),
    ("answerType",pa.string()),
    ("answerTime",pa.string()),
    ("unixTime",pa.int64()),
])

#se qa.parquet esiste gi? lo elimina
if QA_OUT.exists():
    QA_OUT.unlink()

writer=None
buffer=[]
matched_qa_total=0

#teniamo solo domande/risposte dei prodotti presenti anche nei metadati 2023
for category in CATEGORIES:
    label=category["label"]
    category_items=items_id_by_category.get(label,set()) #insieme prodotti della categoria presa
    qa_strict_path=qa_info_by_category[label]["qa_strict_path"] #prende file qa convertito della categoria
    matched_qa=0

    for row in tqdm(iter_jsonl_gz(qa_strict_path),desc=f"Filtro QA {label}"): #legge riga per riga il file qa, con barra di avanzamento tqdm
        parent_asin=row.get("parent_asin")
        if parent_asin not in category_items: #skippa qa di prodotti non presenti in items.parquet
            continue

        buffer.append({ #aggiunge la riga qa al buffer
            "qa_id":int(row.get("qa_id")),
            "parent_asin":parent_asin,
            "source_category":label,
            "hf_category":category["hf_category"],
            "qa_slug":category["qa_slug"],
            "category_group":category["group"],
            "question":row.get("question"),
            "answer":row.get("answer"),
            "questionType":row.get("questionType"),
            "answerType":row.get("answerType"),
            "answerTime":row.get("answerTime"),
            "unixTime":toint_or_none(row.get("unixTime")),
        })
        matched_qa+=1
        matched_qa_total+=1

        if len(buffer)>=10000: #batch piu grande per scrittura QA
            writer=writerows_in_parquet(buffer,QA_OUT,QA_SCHEMA,writer)
            buffer=[]

    for summary_row in category_summary_rows:
        if summary_row["source_category"]==label:
            summary_row["qa_rows_after_metadata"]=int(matched_qa)
            summary_row["qa_products_after_metadata"]=int(len(category_items))
            break
    print(label,"| QA filtrate:",matched_qa)

writer=writerows_in_parquet(buffer,QA_OUT,QA_SCHEMA,writer)
if writer is not None:
    writer.close()

qa_df=pd.read_parquet(QA_OUT) if QA_OUT.exists() else pd.DataFrame(columns=[field.name for field in QA_SCHEMA])
print("QA filtrate totali:",len(qa_df))
display(qa_df.head())


Filtro QA Appliances: 9011it [00:00, 166649.94it/s]


Appliances | QA filtrate: 3826


Filtro QA Arts Crafts and Sewing: 21262it [00:00, 156574.94it/s]


Arts Crafts and Sewing | QA filtrate: 701


Filtro QA Automotive: 89923it [00:00, 176691.32it/s]


Automotive | QA filtrate: 7076


... (33 outputs left out)

Toys and Games | QA filtrate: 3528


Filtro QA Video Games: 13307it [00:00, 150213.84it/s]

Video Games | QA filtrate: 5499
QA filtrate totali: 68871


,qa_id,parent_asin,source_category,hf_category,qa_slug,category_group,question,answer,questionType,answerType,answerTime,unixTime
0,68,B00009V3UA,Appliances,Appliances,Appliances,home,will these range replacement knobs fit a tappa...,It fit my Tappan stove.,yes/no,Y,"Feb 19, 2014",1.392797e+09
1,69,B00009V3UA,Appliances,Appliances,Appliances,home,dear sir how many kilo metres,The bushings that you see are made to fit many...,open-ended,NaN,"Aug 28, 2013",1.377673e+09
2,70,B00009V3UA,Appliances,Appliances,Appliances,home,"What are the different colored round ""things"" ...","They are to make the knobs fit,but the knobs k...",open-ended,NaN,"Apr 16, 2015",1.429168e+09
3,71,B00009V3UA,Appliances,Appliances,Appliances,home,Do you have to hold down the IS button or does...,"They did not work on my stove,",open-ended,NaN,"Mar 30, 2015",1.427699e+09
4,72,B00009V3UA,Appliances,Appliances,Appliances,home,do you have replacement knobs for hotpoint stove?,I am so sorry but at this time i do not. tower...,yes/no,N,"Mar 19, 2015",1.426748e+09


In [11]:
#Scansioniamo reviews 2023 e teniamo solo rating dei prodotti gia selezionati
#Questo completa l'intersezione metadata+QA+rating
#Il file puo essere riusato solo se la firma di build coincide con soglie e item correnti
#Se cambi soglie o prodotti selezionati, ratings.parquet viene ricostruito automaticamente
#creiamo quindi ratings.parquet
#lista di colonne per file di rating
rating_fields=[
    ("user_id",pa.string()),
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("asin",pa.string()),
    ("rating",pa.float64()),
    ("timestamp",pa.int64()),
    ("verified_purchase",pa.bool_()),
]

#se piu avanti vogliamo usare recensioni per sistema qa o per altri sistemi recomendation
if SAVE_TEXT_OF_REVIEW:
    rating_fields.extend([
        ("review_title",pa.string()),
        ("review_text",pa.string()),
    ])

RATING_SCHEMA=pa.schema(rating_fields)

def current_ratings_build_info()->dict:
    return {
        "save_text_of_review":bool(SAVE_TEXT_OF_REVIEW),
        "categories":[category["label"] for category in CATEGORIES],
        "max_reviews_scan_by_category":{key:int(value) if value is not None else None for key,value in MAX_REVIEWS_SCAN_BY_CATEGORY.items()},
        "max_ratings_saved_by_category":{key:int(value) if value is not None else None for key,value in MAX_RATINGS_SAVED_BY_CATEGORY.items()},
        "items_by_category":{
            category["label"]:sorted(items_id_by_category.get(category["label"],set()))
            for category in CATEGORIES
        },
    }

def load_json_or_none(path:Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def update_rating_summary_from_parquet()->None:
    if not RATINGS_OUT.exists() or RATINGS_OUT.stat().st_size<=0:
        return
    ratings_summary_df=pd.read_parquet(RATINGS_OUT,columns=["source_category","parent_asin"])
    rows_by_category=ratings_summary_df.groupby("source_category").size().to_dict()
    products_by_category=ratings_summary_df.groupby("source_category")["parent_asin"].nunique().to_dict()
    for summary_row in category_summary_rows:
        label=summary_row["source_category"]
        summary_row["reviews_scanned"]=None
        summary_row["ratings_saved"]=int(rows_by_category.get(label,0))
        summary_row["rating_products"]=int(products_by_category.get(label,0))

ratings_build_info=current_ratings_build_info()
saved_ratings_build_info=load_json_or_none(RATINGS_BUILD_INFO_PATH)
ratings_cache_valid=False
ratings_cache_reason="ratings.parquet assente o vuoto"

if RATINGS_OUT.is_file() and RATINGS_OUT.stat().st_size>0 and not FORCE_REBUILD_RATINGS:
    if saved_ratings_build_info is None:
        ratings_cache_reason="firma ratings_build_info.json assente o non leggibile"
    elif saved_ratings_build_info != ratings_build_info:
        ratings_cache_reason="soglie o item selezionati cambiati"
    else:
        ratings_cache_valid=True
        ratings_cache_reason="firma coerente"
elif FORCE_REBUILD_RATINGS:
    ratings_cache_reason="FORCE_REBUILD_RATINGS=True"

if ratings_cache_valid:
    n_rows=pq.ParquetFile(RATINGS_OUT).metadata.num_rows
    update_rating_summary_from_parquet()
    print(f"ratings.parquet riusato: {RATINGS_OUT}")
    print(f"Righe presenti: {n_rows:,}")
    print(f"Motivo cache: {ratings_cache_reason}")
else:
    print(f"Ricostruisco ratings.parquet: {ratings_cache_reason}")
    if RATINGS_OUT.exists():
        RATINGS_OUT.unlink()
    if RATINGS_BUILD_INFO_PATH.exists():
        RATINGS_BUILD_INFO_PATH.unlink()

    writer=None
    buffer=[]
    scanned_total=0
    matched_ratings_total=0

    #scansioniamo tutte le reviews
    for category in CATEGORIES: #cicla categorie
        label=category["label"]
        hf_category=category["hf_category"]
        category_items=items_id_by_category.get(label,set())
        max_scan=MAX_REVIEWS_SCAN_BY_CATEGORY.get(label)
        max_saved=MAX_RATINGS_SAVED_BY_CATEGORY.get(label)

        review_stream=load_dataset( #carichi review della categoria corrente
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_review_{hf_category}",
            split="full",
            streaming=True, #legge righe progressivamente
            trust_remote_code=True,
        )

        matched_ratings=0
        scanned=0

        for row in tqdm(review_stream,desc=f"Filtro rating {label}"): #legge una review alla volta
            if max_saved is not None and matched_ratings>=max_saved:
                break
            if max_scan is not None and scanned>=max_scan:
                break
            scanned+=1

            parent_asin=row.get("parent_asin")
            if parent_asin not in category_items: #skippa reviews di prodotti non nel catalogo finale
                continue

            out_row={ #riga rating finale
                "user_id":row.get("user_id"),
                "parent_asin":parent_asin,
                "source_category":label,
                "hf_category":hf_category,
                "qa_slug":category["qa_slug"],
                "category_group":category["group"],
                "asin":row.get("asin"),
                "rating":tofloat_or_none(row.get("rating")),
                "timestamp":toint_or_none(row.get("timestamp")),
                "verified_purchase":row.get("verified_purchase"),
            }

            if SAVE_TEXT_OF_REVIEW: #piu testo se lo vogliamo, per ora no, per alleggerire e testare modello base
                out_row["review_title"]=row.get("title")
                out_row["review_text"]=row.get("text")

            buffer.append(out_row)
            matched_ratings+=1
            matched_ratings_total+=1

            if len(buffer)>=20000: #batch piu grande per molte review
                writer=writerows_in_parquet(buffer,RATINGS_OUT,RATING_SCHEMA,writer)
                buffer=[]

        scanned_total+=scanned
        for summary_row in category_summary_rows:
            if summary_row["source_category"]==label:
                summary_row["reviews_scanned"]=int(scanned)
                summary_row["ratings_saved"]=int(matched_ratings)
                break
        print(label,"| review scansionate:",scanned,"| rating salvati:",matched_ratings)

    writer=writerows_in_parquet(buffer,RATINGS_OUT,RATING_SCHEMA,writer)
    if writer is not None:
        writer.close()

    RATINGS_BUILD_INFO_PATH.write_text(json.dumps(ratings_build_info,indent=2,ensure_ascii=False),encoding="utf-8")
    print("Review scansionate totali:",scanned_total)
    print("Rating salvati totali:",matched_ratings_total)
    print("Firma ratings salvata:",RATINGS_BUILD_INFO_PATH)


Ricostruisco ratings.parquet: firma ratings_build_info.json assente o non leggibile


Filtro rating Appliances: 175000it [00:09, 18059.35it/s]


Appliances | review scansionate: 175000 | rating salvati: 7335


Filtro rating Arts Crafts and Sewing: 100000it [00:06, 16079.47it/s]


Arts Crafts and Sewing | review scansionate: 100000 | rating salvati: 118


Filtro rating Automotive: 350000it [00:21, 16286.48it/s]


... (33 outputs left out)

Filtro rating Toys and Games: 67574it [00:05, 18780.56it/s]'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw/review_categories/Toys_and_Games.jsonl
Retrying in 1s [Retry 1/5].
Filtro rating Toys and Games: 175000it [00:23, 7513.23it/s] 


Toys and Games | review scansionate: 175000 | rating salvati: 1395


Filtro rating Video Games: 175000it [00:11, 14688.45it/s]

Video Games | review scansionate: 175000 | rating salvati: 7675
Review scansionate totali: 4525000
Rating salvati totali: 48557
Firma ratings salvata: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\processed\prototype_amazon_like_intersection\ratings_build_info.json


In [12]:
#Allineiamo items e QA ai soli prodotti che hanno rating
#Questo garantisce che ogni prodotto finale abbia metadata,QA e interazioni
#Dopo il filtro riscriviamo parquet e riepilogo categoria
#allineamento items e qa ai soli prodotti che hanno rating
#cos? il dataset finale contiene prodotti con metadata + QA + rating
ratings_df=pd.read_parquet(RATINGS_OUT,columns=["parent_asin","source_category"]) if RATINGS_OUT.exists() else pd.DataFrame(columns=["parent_asin","source_category"]) #legge solo colonne necessarie al filtro
rating_products=set(ratings_df["parent_asin"]) #insieme di prodotti che hanno almeno un rating

items_df=items_df[items_df["parent_asin"].isin(rating_products)].copy() #filtra items df tenendo solo quelli presenti in ratings products
items_df.to_parquet(ITEMS_OUT,index=False)
items_id=set(items_df["parent_asin"])

valid_products=set(items_df["parent_asin"]) #filtra anche qadf
qa_df=qa_df[qa_df["parent_asin"].isin(valid_products)].copy()
qa_df.to_parquet(QA_OUT,index=False)

#conta prodotti finali per categoria
category_final_counts=(
    items_df.groupby("source_category")["parent_asin"]
    .nunique()
    .rename("final_products_qa_metadata_rating")
    .reset_index()
)

for summary_row in category_summary_rows: #aggiorna riepilogo per categoria, aggiungendo num prodotti finali
    label=summary_row["source_category"]
    final_count=category_final_counts.loc[
        category_final_counts["source_category"]==label,
        "final_products_qa_metadata_rating",
    ]
    summary_row["final_products_qa_metadata_rating"]=int(final_count.iloc[0]) if len(final_count) else 0

category_summary_df=pd.DataFrame(category_summary_rows)
category_summary_df.to_csv(CATEGORY_SUMMARY_PATH,index=False)

print("Prodotti finali con metadata + QA + rating:",len(items_df))
print("Prodotti con QA dopo filtro:",qa_df["parent_asin"].nunique() if len(qa_df) else 0)
print("Righe QA dopo filtro:",len(qa_df))
print("Riepilogo per categoria:",CATEGORY_SUMMARY_PATH)
display(category_summary_df)


Prodotti finali con metadata + QA + rating: 4864
Prodotti con QA dopo filtro: 4864
Righe QA dopo filtro: 40713
Riepilogo per categoria: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\processed\prototype_amazon_like_intersection\category_summary.csv


,source_category,hf_category,qa_slug,category_group,qa_rows,qa_products,metadata_rows,items_qa_metadata,duplicate_items_skipped,qa_rows_after_metadata,qa_products_after_metadata,reviews_scanned,ratings_saved,final_products_qa_metadata_rating
0,Appliances,Appliances,Appliances,home,9011,1153,67424,500,0,3826,500,175000,7335,415
1,Arts Crafts and Sewing,Arts_Crafts_and_Sewing,Arts_Crafts_and_Sewing,crafts,21262,2944,58189,100,0,701,100,100000,118,43
2,Automotive,Automotive,Automotive,automotive,89923,12536,218637,1000,0,7076,1000,350000,2222,560
3,Baby Products,Baby_Products,Baby,baby,28933,3783,59199,500,0,3882,500,175000,3103,334
4,Beauty,All_Beauty,Beauty,beauty,42422,5906,112590,47,0,287,47,175000,1285,46
5,Cell Phones and Accessories,Cell_Phones_and_Accessories,Cell_Phones_and_Accessories,tech,85865,10407,125000,451,0,3791,451,175000,566,176
6,Clothing Shoes and Jewelry,Clothing_Shoes_and_Jewelry,Clothing_Shoes_and_Jewelry,fashion,22068,2899,250000,19,0,131,19,350000,10,4
7,Electronics,Electronics,Electronics,tech,314263,39371,71438,1000,0,7768,1000,350000,1339,357
8,Grocery and Gourmet Food,Grocery_and_Gourmet_Food,Grocery_and_Gourmet_Food,grocery,19538,3017,125000,273,0,1775,273,175000,456,140
9,Health and Personal Care,Health_and_Personal_Care,Health_and_Personal_Care,health,80496,10860,60293,89,0,623,89,350000,10705,89


In [13]:
#dizionario riassuntivo
summary={
    "dataset_name":DATASET_NAME,
    "dataset_label":DATASET_LABEL,
    "categories":[category["label"] for category in CATEGORIES],
    "items":parquet_count_rows(ITEMS_OUT) if ITEMS_OUT.exists() else 0,
    "qa":parquet_count_rows(QA_OUT) if QA_OUT.exists() else 0,
    "ratings":parquet_count_rows(RATINGS_OUT) if RATINGS_OUT.exists() else 0,
    "processed_dir":str(PROCESSED_DIR),
    "items_path":str(ITEMS_OUT),
    "qa_path":str(QA_OUT),
    "ratings_path":str(RATINGS_OUT),
    "ratings_build_info_path":str(RATINGS_BUILD_INFO_PATH),
    "category_summary_path":str(CATEGORY_SUMMARY_PATH),
}

SUMMARY_PATH.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
summary


{'dataset_name': 'prototype_amazon_like_intersection',
 'dataset_label': 'AmazonLikePrototypeMini',
 'categories': ['Appliances',
  'Arts Crafts and Sewing',
  'Automotive',
  'Baby Products',
  'Beauty',
  'Cell Phones and Accessories',
  'Clothing Shoes and Jewelry',
  'Electronics',
  'Grocery and Gourmet Food',
  'Health and Personal Care',
  'Home and Kitchen',
  'Industrial and Scientific',
  'Musical Instruments',
  'Office Products',
  'Patio Lawn and Garden',
  'Pet Supplies',
  'Software',
  'Sports and Outdoors',
  'Tools and Home Improvement',
  'Toys and Games',
  'Video Games'],
 'items': 4864,
 'qa': 40713,
 'ratings': 48557,
 'processed_dir': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\prototype\\data\\processed\\prototype_amazon_like_intersection',
 'items_path': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\prototype\\data\\processed\\prototype_amazon_like_intersection\\items.parquet',
 'qa_path': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\prototype\\data\\processed\\p

In [14]:
#Prepariamo export products_demo.json per il frontend
#Convertiamo campi parquet complessi in testo e valori JSON validi
#Il catalogo demo viene ordinato e bilanciato per categoria
#proviamo a leggere stringhe JSON salvate nei parquet
def parse_json_value(value):
    if value in (None,"","null","None"):
        return None
    if not isinstance(value,str):
        return value
    try:
        return json.loads(value)
    except Exception:
        return value

#convertiamo descrizioni,features o details in testo leggibile
def readable_text(*values)->str:
    for value in values:
        parsed=parse_json_value(value)
        if isinstance(parsed,list):
            parts=[str(item).strip() for item in parsed if str(item).strip()]
            if parts:
                return " ".join(parts)
        if isinstance(parsed,dict):
            parts=[]
            for key,item in parsed.items():
                if isinstance(item,(str,int,float)) and not pd.isna(item):
                    parts.append(f"{key}: {item}")
            if parts:
                return " ".join(parts)
        if isinstance(parsed,str) and parsed.strip():
            return parsed.strip()
    return ""

#convertiamo NaN pandas in None prima di scrivere JSON
def clean_json_value(value):
    if isinstance(value,dict):
        return {key:clean_json_value(item) for key,item in value.items()}
    if isinstance(value,list):
        return [clean_json_value(item) for item in value]
    try:
        if pd.isna(value):
            return None
    except (TypeError,ValueError):
        pass
    return value

#carichiamo items.parquet e prepariamo il catalogo demo del sito
items_for_demo=pd.read_parquet(ITEMS_OUT) if ITEMS_OUT.exists() else pd.DataFrame()

if len(items_for_demo):
    demo=items_for_demo.copy()
    demo["description"]=demo.apply(
        lambda row: readable_text(row.get("description_json"),row.get("features_json"),row.get("details_json")),
        axis=1,
    )
    demo["item_id"]=demo["parent_asin"]
    demo["category"]=demo["source_category"]

    if "avg_rating" in demo.columns:
        demo["average_rating"]=demo["avg_rating"]
    elif "average_rating" not in demo.columns:
        demo["average_rating"]=0.0

    if "rating_number" in demo.columns:
        demo["rating_count"]=demo["rating_number"].fillna(0).astype(int)
    elif "num_ratings" in demo.columns:
        demo["rating_count"]=demo["num_ratings"].fillna(0).astype(int)
    else:
        demo["rating_count"]=0

    max_rating_count=demo["rating_count"].max()
    if max_rating_count and max_rating_count>0:
        demo["popularity_score"]=(demo["rating_count"] / max_rating_count).round(4)
    else:
        demo["popularity_score"]=0.0

    demo=demo.sort_values(["category","popularity_score"],ascending=[True,False]) #ordina prima categoria,poi prodotto piu popolare
    demo_records=demo[
        [
            "item_id",
            "title",
            "category",
            "description",
            "price",
            "average_rating",
            "rating_count",
            "popularity_score",
            "image_url",
        ]
    ].to_dict(orient="records")
    demo_records=clean_json_value(demo_records)
else:
    demo_records=[]

#salviamo products_demo.json senza ulteriore taglio globale
PRODUCTS_DEMO_PATH.write_text(json.dumps(demo_records,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
print("products_demo.json:",PRODUCTS_DEMO_PATH)
print("Prodotti demo:",len(demo_records))


products_demo.json: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\products_demo.json
Prodotti demo: 4864


In [15]:
#controlli finali richiesti
final_items=pd.read_parquet(ITEMS_OUT) if ITEMS_OUT.exists() else pd.DataFrame()
final_qa=pd.read_parquet(QA_OUT) if QA_OUT.exists() else pd.DataFrame()
final_ratings_count=parquet_count_rows(RATINGS_OUT) if RATINGS_OUT.exists() else 0
final_demo=pd.read_json(PRODUCTS_DEMO_PATH) if PRODUCTS_DEMO_PATH.exists() and PRODUCTS_DEMO_PATH.stat().st_size>2 else pd.DataFrame()

print("Numero prodotti totali:",len(final_items))
print("Numero categorie finali:",final_items["source_category"].nunique() if len(final_items) else 0)
print("Numero QA:",len(final_qa))
print("Numero ratings:",final_ratings_count)
print("Percorso products_demo.json:",PRODUCTS_DEMO_PATH)

print("Prodotti per categoria:")
display(category_summary_df[["source_category","final_products_qa_metadata_rating"]])

print("Prime 10 righe catalogo demo:")
display(final_demo.head(10))

Numero prodotti totali: 4864
Numero categorie finali: 21
Numero QA: 40713
Numero ratings: 48557
Percorso products_demo.json: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\products_demo.json
Prodotti per categoria:


,source_category,final_products_qa_metadata_rating
0,Appliances,415
1,Arts Crafts and Sewing,43
2,Automotive,560
3,Baby Products,334
4,Beauty,46
5,Cell Phones and Accessories,176
6,Clothing Shoes and Jewelry,4
7,Electronics,357
8,Grocery and Gourmet Food,140
9,Health and Personal Care,89


Prime 10 righe catalogo demo:


,item_id,title,category,description,price,average_rating,rating_count,popularity_score,image_url
0,B002JAKRAM,Frigidaire ULTRAWF PureSource Ultra Water and ...,Appliances,"Product Description Keep safe, great tasting w...",48.99,4.7,41130,0.3038,https://m.media-amazon.com/images/I/61fOUFc2KN...
1,B000AST3AK,GE MWF Refrigerator Water Filter | Certified t...,Appliances,Product Description GE�s NSF-certified MWF rep...,46.98,4.7,21414,0.1582,https://m.media-amazon.com/images/I/714Di-HcZE...
2,B00AF7U4C2,LG LT700P- 6 Month / 200 Gallon Capacity Repla...,Appliances,Product Description Keeping your refrigerator ...,47.86,4.7,9659,0.0714,https://m.media-amazon.com/images/I/71pBVp8IZC...
3,B0042ACZU2,LG LT700P- 6 Month / 200 Gallon Capacity Repla...,Appliances,Product Description Keeping your refrigerator ...,46.74,4.7,9567,0.0707,https://m.media-amazon.com/images/I/71pBVp8IZC...
4,B000DZFTC6,Dundas Jafine TDIDVKZW Indoor Dryer Vent Kit w...,Appliances,Product Description Dundas Jafine TDIDVKZW Pro...,15.99,4.3,8160,0.0603,https://m.media-amazon.com/images/I/91gsywwjdU...
5,B001ICYB2M,SAMSUNG Heating Element Dc47-00019A,Appliances,This high quality Genuine OEM Samsung Heating ...,NaN,4.6,6451,0.0477,https://m.media-amazon.com/images/I/81cU4gWsTn...
6,B00KJ07SEM,Waterfall Filter - Refrigerator Water Filter C...,Appliances,Waterfall Filter - Refrigerator Water Filter C...,24.99,4.6,4349,0.0321,https://m.media-amazon.com/images/I/712ov0QgkD...
7,B0032AMC9C,LG LT500P - 6 Month / 500 Gallon Capacity Repl...,Appliances,Product Description The LG LT500P Replacement ...,45.84,4.8,3231,0.0239,https://m.media-amazon.com/images/I/71ImfMurhc...
8,B00DM8K72S,ERP 240323001 Refrigerator Door Bin,Appliances,Product Description This is a genuine replacem...,NaN,4.6,2777,0.0205,https://m.media-amazon.com/images/I/41q8l9i62p...
9,B0053F7UIM,Whirlpool 3387747 Element for Dryer,Appliances,Product Description This is a genuine replacem...,NaN,4.4,2699,0.0199,https://m.media-amazon.com/images/I/51rTz-EBbe...


In [16]:
#Controlliamo quanti prodotti demo non hanno prezzo
#Serve per capire se il frontend puo mostrare card complete
#Il riepilogo per categoria aiuta a individuare categorie problematiche
#controlliamo quanti prodotti demo non hanno prezzo
if PRODUCTS_DEMO_PATH.exists():
    price_check_df=pd.read_json(PRODUCTS_DEMO_PATH)
elif ITEMS_OUT.exists():
    price_check_df=pd.read_parquet(ITEMS_OUT)
else:
    price_check_df=pd.DataFrame()

if len(price_check_df):
    missing_price=price_check_df["price"].isna().sum() if "price" in price_check_df.columns else len(price_check_df)
    total_items=len(price_check_df)
    missing_price_percentage=round((missing_price / total_items) * 100,2) #percentuale leggibile per report
    with_price=total_items - missing_price

    print("Prodotti totali:",total_items)
    print("Prodotti con prezzo:",with_price)
    print("Prodotti senza prezzo:",missing_price)
    print("Percentuale senza prezzo:",f"{missing_price_percentage}%")

    if "category" in price_check_df.columns:
        price_missing_by_category=(
            price_check_df
            .assign(price_missing=price_check_df["price"].isna() if "price" in price_check_df.columns else True)
            .groupby("category")
            .agg(
                products=("category","size"),
                missing_price=("price_missing","sum"),
            )
            .reset_index()
        )
        price_missing_by_category["missing_price_percentage"]=(
            price_missing_by_category["missing_price"] / price_missing_by_category["products"] * 100
        ).round(2)
        display(price_missing_by_category.sort_values("missing_price_percentage",ascending=False))
else:
    print("Nessun prodotto disponibile per il controllo prezzi")

Prodotti totali: 4864
Prodotti con prezzo: 2242
Prodotti senza prezzo: 2622
Percentuale senza prezzo: 53.91%


,category,products,missing_price,missing_price_percentage
5,Cell Phones and Accessories,176,159,90.34
16,Software,68,61,89.71
3,Baby Products,334,275,82.34
4,Beauty,46,32,69.57
12,Musical Instruments,124,81,65.32
7,Electronics,357,228,63.87
15,Pet Supplies,35,22,62.86
9,Health and Personal Care,89,52,58.43
17,Sports and Outdoors,466,267,57.30
10,Home and Kitchen,228,127,55.70
